In [ ]:
%pip install gurobipy

In [ ]:
import pandas as pd
from itertools import product

import gurobipy as gp
from gurobipy import GRB

# tested with Python 3.11 & Gurobi 11.0

## Input data

We define all the input data for the model.

In [ ]:
# list of depots and working days of a week

depots = ['Glasgow','Manchester','Birmingham','Plymouth']
NRD = ['Glasgow','Plymouth'] # Non-repair depot
RD =['Manchester','Birmingham'] # Repair depot

days = [0,1,2,3,4,5] # Monday = 0, Tuesday = 1, ...  Saturday = 5
rentDays = [1,2,3]

d2w, demand = gp.multidict({
    ('Glasgow',0): 100,
    ('Glasgow',1): 150,
    ('Glasgow',2): 135,
    ('Glasgow',3): 83,
    ('Glasgow',4): 120,
    ('Glasgow',5): 230,
    ('Manchester',0): 250,
    ('Manchester',1): 143,
    ('Manchester',2): 80,
    ('Manchester',3): 225,
    ('Manchester',4): 210,
    ('Manchester',5): 98,
    ('Birmingham',0): 95,
    ('Birmingham',1): 195,
    ('Birmingham',2): 242,
    ('Birmingham',3): 111,
    ('Birmingham',4): 70,
    ('Birmingham',5): 124,
    ('Plymouth',0): 160,
    ('Plymouth',1): 99,
    ('Plymouth',2): 55,
    ('Plymouth',3): 96,
    ('Plymouth',4): 115,
    ('Plymouth',5): 80
})

#repairCap
depots, capacity = gp.multidict({
    ('Glasgow'): 0,
    ('Manchester'): 12,
    ('Birmingham'): 20,
    ('Plymouth'): 0
})

# Create a dictionary to capture
# pctRent: percentage of cars rented for r days
# cstMarginal: marginal cost for renting a car for r days
# prcSameD: price of renting a car r days and returning to same depot
# prcOtherD: price of renting a car r days and returning to another depot
rentDays, pctRent, costMarginal, priceSameD, priceOtherD = gp.multidict({
    (1): [0.55,20,50,70],
    (2): [0.20,25,70,100],
    (3): [0.25,30,120,150]
})

# Cost of owing a car per week.
cstOwn = 15

# Proportional damaged car fee
damagedFee = 10

# Create a dictionary to capture the proportion of cars rented at depot d to be returned to depot d2
d2d, pctFromToD = gp.multidict({
    ('Glasgow','Glasgow'): 0.6,
    ('Glasgow','Manchester'): 0.2,
    ('Glasgow','Birmingham'): 0.1,
    ('Glasgow','Plymouth'): 0.1,
    ('Manchester','Glasgow'): 0.15,
    ('Manchester','Manchester'): 0.55,
    ('Manchester','Birmingham'): 0.25,
    ('Manchester','Plymouth'): 0.05,
    ('Birmingham','Glasgow'): 0.15,
    ('Birmingham','Manchester'): 0.2,
    ('Birmingham','Birmingham'): 0.54,
    ('Birmingham','Plymouth'): 0.11,
    ('Plymouth','Glasgow'): 0.08,
    ('Plymouth','Manchester'): 0.12,
    ('Plymouth','Birmingham'): 0.27,
    ('Plymouth','Plymouth'): 0.53
})

# Create a dictionary to capture the transfer costs  of cars
d2d, cstFromToD = gp.multidict({
    ('Glasgow','Glasgow'): 0.001,
    ('Glasgow','Manchester'): 20,
    ('Glasgow','Birmingham'): 30,
    ('Glasgow','Plymouth'): 50,
    ('Manchester','Glasgow'): 20,
    ('Manchester','Manchester'): 0.001,
    ('Manchester','Birmingham'): 15,
    ('Manchester','Plymouth'): 35,
    ('Birmingham','Glasgow'): 30,
    ('Birmingham','Manchester'): 15,
    ('Birmingham','Birmingham'): 0.001,
    ('Birmingham','Plymouth'): 25,
    ('Plymouth','Glasgow'): 50,
    ('Plymouth','Manchester'): 35,
    ('Plymouth','Birmingham'): 25,
    ('Plymouth','Plymouth'): 0.001
})

# Proportion of undamaged and damaged cars returned
pctUndamaged = 0.9
pctDamaged = 0.1


### Preprocessing
We prepare the data structures to build the linear programming model.

In [ ]:
# Build a list of tuples (depot, depot2) such that d != d2
list_d2notd = []

for d,d2 in d2d:
    if (d != d2):
        tp = d,d2
        list_d2notd.append(tp)

d2notd = gp.tuplelist(list_d2notd)

# Build a list of tuples (depot, depot2, day)
list_dd2t = []

for d,d2 in d2notd:
    for t in days:
        tp = d,d2,t
        list_dd2t.append(tp)

dd2t = gp.tuplelist(list_dd2t)

# Build a list of tuples (depot, rent_day)
list_dr = []

for d in depots:
    for r in rentDays:
        tp = d,r
        list_dr.append(tp)

dr = gp.tuplelist(list_dr)

# Build a list of tuples (depot, day, rent_days )
list_dtr = []

for d in depots:
    for t in days:
            for r in rentDays:
                tp = d,t,r
                list_dtr.append(tp)

dtr = gp.tuplelist(list_dtr)

# Build a list of tuples (depot, depot2, day, rent_days)
list_dd2tr = []

for d,d2 in d2notd:
    for t in days:
        for r in rentDays:
            tp = d,d2,t,r
            list_dd2tr.append(tp)


dd2tr = gp.tuplelist(list_dd2tr)

## Model Deployment
We create a model and the variables. The main decision variables are the number of cars to own
and where should they be located at the start of each day of a week to maximize weekly profits.

In [ ]:
model = gp.Model('RentalCar1')

# Number of cars owned
n = model.addVar(name="cars")

# Number of undamaged cars
nu = model.addVars(d2w, name="UDcars")

# Number of damaged cars
nd = model.addVars(d2w, name="Dcars")

# Number of cars hired (rented) cannot exceed their demand
tr = model.addVars(d2w, ub=demand, name="Hcars")
#for d,t in d2w:
    #tr[d,t].lb = 1

# End inventory of undamaged cars
eu = model.addVars(d2w, name="EUDcars")

# End inventory of damaged cars
ed = model.addVars(d2w, name="EDcars")

# Number of undamaged cars transferred
tu = model.addVars(dd2t, name="TUDcars")

# Number of damaged cars transferred
td = model.addVars(dd2t, name="TDcars")

# Number of damaged cars repaired
rp = model.addVars(d2w, name="RPcars")

# Number of damaged cars repaired cannot exceed depot capacity
for d,t in d2w:
    rp[d,t].ub = capacity[d] #repair capacity

Using license file c:\gurobi\gurobi.lic


### Constraints
The number of undamaged cars available at a non-repair depot d at the beginning of  day t should be equal to the demand of undamaged cars at the non-repair depot d during day t.

In [ ]:
# Undamaged cars into a non-repair depot constraints (left hand side of balance equation -availability)

UDcarsNRD_L = model.addConstrs((gp.quicksum(pctUndamaged*pctFromToD[d2,d]*pctRent[r]*tr[d2,(t-r)%6 ] for d2,r in dr )
                              + gp.quicksum(tu.select('*',d,(t-1)%6)  )
                              + eu[d,(t-1)%6 ] == nu[d,t] for d in NRD for t in days ),
                             name="UDcarsNRD_L")

# Undamaged cars out of a non-repair depot constraints (right hand side of balance equation -requirements)

UDcarsNRD_R = model.addConstrs((tr[d,t]
                                + gp.quicksum(tu.select(d,'*',t ))
                                + eu[d,t] == nu[d,t] for d in NRD for t in days ), name='UDcarsNRD_R' )

The number of undamaged cars available at a repair depot d at the beginning of  day t should be equal to the demand of undamaged cars at the repair depot d during day t.

In [ ]:
# Undamaged cars into a repair depot constraints (left hand side of balance equation -availability)

UDcarsRD_L = model.addConstrs((gp.quicksum(pctUndamaged*pctFromToD[d2,d]*pctRent[r]*tr[d2,(t-r)%6 ] for d2,r in dr )
                              + gp.quicksum(tu.select('*',d,(t-1)%6)  ) + rp[d, (t-1)%6 ]
                              + eu[d,(t-1)%6 ] == nu[d,t] for d in RD for t in days ),
                             name="UDcarsRD_L")

# Undamaged cars out of a repair depot constraints (right hand side of balance equation -requirements)

UDcarsRD_R = model.addConstrs((tr[d,t]
                                + gp.quicksum(tu.select(d,'*',t ) )
                                + eu[d,t] == nu[d,t] for d in RD for t in days ), name='UDcarsRD_R' )

The number of damaged cars available at a non-repair depot d at the beginning of  day t should be equal to the demand of damaged cars at the non-repair depot d during day t.

In [ ]:
# Damaged cars into a non-repair depot constraints (left hand side of balance equation -availability)

DcarsNRD_L = model.addConstrs((gp.quicksum(pctDamaged*pctFromToD[d2,d]*pctRent[r]*tr[d2,(t-r)%6 ] for d2,r in dr )
                              + ed[d,(t-1)%6 ] == nd[d,t] for d in NRD for t in days ),
                             name="DcarsNRD_L")

# Damaged cars out of a non-repair depot constraints (right hand side of balance equation -requirements)

DcarsNRD_R = model.addConstrs(( gp.quicksum(td[d,d2,t] for d2 in RD )
                                + ed[d,t] == nd[d,t] for d in NRD for t in days ), name='DcarsNRD_R' )

The number of damaged cars available at a repair depot d at the beginning of  day t should be equal to the demand of damaged cars at the repair depot d during day t.

In [ ]:
# Damaged cars into a repair depot constraints (left hand side of balance equation -availability)

DcarsRD_L = model.addConstrs((gp.quicksum(pctDamaged*pctFromToD[d2,d]*pctRent[r]*tr[d2,(t-r)%6 ] for d2,r in dr )
                              + gp.quicksum(td[d2,d,(t-1)%6 ] for d2, dd in d2notd if (dd == d))
                              + ed[d,(t-1)%6 ] == nd[d,t] for d in RD for t in days ),
                             name="DcarsRD_L")

# Damaged cars out of a repair depot constraints (right hand side of balance equation -requirements)

DcarsND_R = model.addConstrs((rp[d,t] + gp.quicksum(td[d,d2,t ] for d2 in NRD )
                                + ed[d,t] == nd[d,t] for d in RD for t in days ), name='DcarsND_R' )

Total number of cars equals the number of cars rented out from all depots on Monday for 3 days, plus those on Tuesday for 2 or 3 days, plus all damaged and undamaged cars in depots at the beginning of Wednesday.

In [ ]:
# Total number of cars owned constraint
# Note: 25% of cars are rented for 3 days, and 20% + 25% = 45% of the cars are rented for 2-days or 3-days

carsConstr = model.addConstr((gp.quicksum(0.25*tr[d,0] + 0.45*tr[d,1] + nu[d,2] + nd[d,2] for d in depots )
                              == n ),name='carsConstr')

The objective function is to maximize profit.

In [ ]:
# Maximize profit objective function

model.setObjective((
    gp.quicksum(pctFromToD[d,d]*pctRent[r]*(priceSameD[r] - costMarginal[r] + damagedFee)*tr[d,t] for d,t,r in dtr )
    + gp.quicksum(pctFromToD[d,d2]*pctRent[r]*(priceOtherD[r]-costMarginal[r]+damagedFee)*tr[d,t] for d,d2,t,r in dd2tr)
    - gp.quicksum(cstFromToD[d,d2]*tu[d,d2,t] for d,d2,t in dd2t)
    - gp.quicksum(cstFromToD[d,d2]*td[d,d2,t] for d,d2,t in dd2t) - cstOwn*n ), GRB.MAXIMIZE)

In [ ]:
# Verify model formulation

model.write('CarRental1.lp')

# Run optimization engine

model.optimize()

Gurobi Optimizer version 9.1.0 build v9.1.0rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 97 rows, 289 columns and 1061 nonzeros
Model fingerprint: 0x7ddb81fe
Coefficient statistics:
  Matrix range     [1e-03, 1e+00]
  Objective range  [2e+01, 7e+01]
  Bounds range     [1e+01, 3e+02]
  RHS range        [0e+00, 0e+00]
Presolve removed 49 rows and 85 columns
Presolve time: 0.01s
Presolved: 48 rows, 204 columns, 936 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.1102412e+05   3.626166e+02   0.000000e+00      0s
      69    1.2116021e+05   0.000000e+00   0.000000e+00      0s

Solved in 69 iterations and 0.01 seconds
Optimal objective  1.211602072e+05


---
## Analysis

In [ ]:
# Output report

# Total number of cars owned
print(f"The optimal number of cars to be owned is: {round(n.x)}.")

# Optimal profit
print(f"The optimal profit is: {'${:,.2f}'.format(round(model.objVal,2))}.")


The optimal number of cars to be owned is: 617.
The optimal profit is: $121,160.21.


In [ ]:
# Create a list to translate the number label of each day to the actual name of the day
dayname = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']

# Number of undamaged cars in depot at the beginning of each day.
print("\n\n_________________________________________________________________________________")
print(f"Estimated number of undamaged cars in depot at the beginning of each day: ")
print("_________________________________________________________________________________")

undamaged_cars = pd.DataFrame(
    {
        "Day": [dayname[t] for t in days],
        "Glasgow": [round(nu['Glasgow',t].x) for t in days],
        "Manchester": [round(nu['Manchester',t].x) for t in days],
        "Birmingham": [round(nu['Birmingham',t].x) for t in days],
        "Plymouth": [round(nu['Plymouth',t].x) for t in days],
    }
)
undamaged_cars.index=[''] * len(undamaged_cars)
undamaged_cars



_________________________________________________________________________________
Estimated number of undamaged cars in depot at the beginning of each day: 
_________________________________________________________________________________


,Day,Glasgow,Manchester,Birmingham,Plymouth
,Monday,68,98,146,41
,Tuesday,66,95,155,40
,Wednesday,70,100,123,43
,Thursday,68,114,116,42
,Friday,70,102,124,43
,Saturday,67,95,158,40


In [ ]:
# Number of Damaged cars in depot at the beginning of each day.
print("_________________________________________________________________________________")
print(f"Estimated number of damaged cars in depot at the beginning of each day: ")
print("_________________________________________________________________________________")

damaged_cars = pd.DataFrame(
    {
        "Day": [dayname[t] for t in days],
        "Glasgow": [round(nd['Glasgow',t].x) for t in days],
        "Manchester": [round(nd['Manchester',t].x) for t in days],
        "Birmingham": [round(nd['Birmingham',t].x) for t in days],
        "Plymouth": [round(nd['Plymouth',t].x) for t in days],
    }
)
damaged_cars.index=[''] * len(damaged_cars)
damaged_cars

_________________________________________________________________________________
Estimated number of damaged cars in depot at the beginning of each day: 
_________________________________________________________________________________


,Day,Glasgow,Manchester,Birmingham,Plymouth
,Monday,8,12,20,6
,Tuesday,7,12,20,4
,Wednesday,8,13,20,5
,Thursday,8,12,21,5
,Friday,8,12,20,7
,Saturday,7,12,22,4


In [ ]:
# Undamaged car rented out from each depot and day.
print("_________________________________________________________________________________")
print(f"Estimated number of undamaged cars rented out from each depot and day: ")
print("_________________________________________________________________________________")

rentedOut = {}

for d in depots:
    for t in days:
        count = 0
        for d2 in depots:
            for r in rentDays:
                #print(f"Depot {d}, day {t}: cars rented out {tr[d,t].x}")
                count += pctUndamaged*pctFromToD[d,d2]*pctRent[r]*tr[d,t].x
        rentedOut[d,t] = round(count)


#print(rentedOut)

rentout_cars = pd.DataFrame(
    {
        "Day": [dayname[t] for t in days],
        "Glasgow": [round(rentedOut['Glasgow',t]) for t in days],
        "Manchester": [round(rentedOut['Manchester',t]) for t in days],
        "Birmingham": [round(rentedOut['Birmingham',t]) for t in days],
        "Plymouth": [round(rentedOut['Plymouth',t]) for t in days],
    }
)
rentout_cars.index=[''] * len(rentout_cars)
rentout_cars

_________________________________________________________________________________
Estimated number of undamaged cars rented out from each depot and day: 
_________________________________________________________________________________


,Day,Glasgow,Manchester,Birmingham,Plymouth
,Monday,61,89,86,37
,Tuesday,59,85,140,36
,Wednesday,63,72,111,39
,Thursday,62,103,100,38
,Friday,63,92,63,39
,Saturday,60,85,112,36
